In [1]:
import osprey
osprey.start()
import osprey.prep

OSPREY 3.3-dev, Python 3.10.15, Java 17.0.2, Linux-5.14.0-503.14.1.el9_5.x86_64-x86_64-with-glibc2.34
Using up to 1024 MiB heap memory: 128 MiB for garbage, 896 MiB for storage


In [2]:
osprey.__file__

'/hpc/group/biostat/etm33/miniconda3/envs/AmberTools22/lib/python3.10/site-packages/osprey/__init__.py'

In [3]:
import os
os.getcwd()

'/hpc/home/etm33/kinase_inhibitor_design'

In [4]:
pdb_path = '/hpc/home/etm33/kinase_inhibitor_design/structures/pkn2_dephos_test.renum.pdb'
pdb = osprey.prep.loadPDB(open(pdb_path, 'r').read())

In [5]:
print('Loaded %d molecules:' % len(pdb))
for mol in pdb:
    print('\t%s: %s' % (mol, osprey.prep.molTypes(mol)))

Loaded 2 molecules:
	Chain A: [Protein]
	Chain E: [Protein]


In [6]:
# looks like the PDB file has a protein chain and a small molecule
# the protein chain must be PTPase, and the small molecule must be HEPES
# (ignore the solvent molecules, if any)
target = pdb[0]
ligand = pdb[1]
mols = [target, ligand]

In [7]:
# start the local service that calls AmberTools for us
# NOTE: this will only work on Linux machines
with osprey.prep.LocalService():

    # Molecule Preparation Step 1: remove duplicate atoms
    # Duplicate atoms don't usually appear in files from the PDB,
    # but these errors can happen sometimes in modified PDB files.
    for mol in mols:
        # remove all but the first duplicated atom from each group
        for group in osprey.prep.duplicateAtoms(mol):
            for atomi in range(1, len(group.getAtoms())):
                group.remove(atomi)
                print('removed duplicate atom %s' % group)
    
    # Molecule Preparation Step 2: add missing heavy atoms
    # Somtimes atom positions are not well-resolved in the electron density,
    # or protein chain end-caps are missing.
    # But we still want to include these atoms in the molecular models to
    # be able to infer bonds correctly in the next step.
    for mol in mols:
        for missing_atom in osprey.prep.inferMissingAtoms(mol):
            missing_atom.add()
            print('added missing atom: %s' % missing_atom)

    # Molecule Preparation Step 3: add bonds
    # PDB files contain no explicit information about bonds, so we have to
    # infer where they might be based on the atoms we can see.
    for mol in mols:
        bonds = osprey.prep.inferBonds(mol)
        for bond in bonds:
            mol.getBonds().add(bond)
        print('added %d bonds to %s' % (len(bonds), mol))

    for mol in mols:
        protonated_atoms = osprey.prep.inferProtonation(mol)
        for protonated_atom in protonated_atoms:
            protonated_atom.add()
        print('added %d hydrogens to %s' % (len(protonated_atoms), mol))

    # Moleclue Preparation Step 7: save the results
    ligand_path = 'pkn2_dephos-ligand.pdb'
    target_path = 'pkn2_dephos-target.pdb'
    mols_path = 'pkn2_dephos-complex.pdb'
    open(ligand_path, 'w').write(osprey.prep.savePDB(ligand))
    print('saved prepared PDB to %s' % ligand_path)
    open(target_path, 'w').write(osprey.prep.savePDB(target))
    print('saved prepared PDB to %s' % target_path)
    # No need to do this with entire molecule?
    #open(mols_path, 'w').write(osprey.prep.savePDB(pdb))
    #print('saved prepared PDB to %s' % pdb)    

print('Moleclue preparation complete!')

14:56:27.454 [main] INFO  ktor.application - Autoreload is disabled because the development mode is off.
14:56:27.477 [main] INFO  ktor.application - Responding at http://0.0.0.0:44342
Osprey prep local service started


log4j:WARN No appenders could be found for logger (org.apache.http.impl.nio.client.MainClientExec).
log4j:WARN Please initialize the log4j system properly.


added 5441 bonds to Chain A
added 247 bonds to Chain E
added 2658 hydrogens to Chain A
added 125 hydrogens to Chain E
saved prepared PDB to pkn2_dephos-ligand.pdb
saved prepared PDB to pkn2_dephos-target.pdb
Osprey prep local service stopped
Moleclue preparation complete!
